# Python 0 — análise de dados aplicada à Bioinformática

**Público:** estudantes sem experiência prévia em Python.  
**Objetivo:** transformar uma base CSV em tabelas, resumos e gráficos.

> Todos os dados e sequências deste material são **sintéticos e didáticos**. Os resultados não sustentam inferências clínicas ou biológicas reais.

## 1. Variáveis, leitura e impressão
Uma variável guarda um valor. `input()` realiza a leitura e `print()` mostra uma saída.

In [ ]:
nome = input('Digite seu nome: ')
area = input('Digite sua área de estudo: ')
numero_amostras = 24

print('Olá,', nome)
print('Área:', area)
print('Número de amostras da prática:', numero_amostras)

## 2. Listas e dicionários
A lista guarda vários valores em ordem. O dicionário relaciona chaves e valores.

In [ ]:
bases_dna = ['A', 'T', 'G', 'C']
amostra = {
    'id': 'A01',
    'grupo': 'Controle',
    'sequencia': 'ATGCGTACGTAA'
}

print('Bases:', bases_dna)
print('Sequência da amostra:', amostra['sequencia'])

for base in bases_dna:
    print('Base:', base)

## 3. Importar as bibliotecas
`pandas` trabalha com tabelas. `matplotlib` cria gráficos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print('Bibliotecas carregadas.')

## 4. Ler o arquivo CSV
Mantenha o CSV na mesma pasta do notebook. No Colab, a célula permitirá selecionar o arquivo.

In [ ]:
from pathlib import Path

arquivo = Path('base_python0_bioinformatica.csv')

if not arquivo.exists():
    try:
        from google.colab import files
        enviados = files.upload()
        arquivo = Path(next(iter(enviados)))
    except ImportError:
        raise FileNotFoundError('Coloque o CSV na mesma pasta do notebook.')

tabela = pd.read_csv(arquivo)
print('Arquivo carregado:', arquivo.name)

## 5. Conhecer a base
Antes de analisar, observe as linhas, colunas, tipos e valores ausentes.

In [ ]:
display(tabela.head())
print('Linhas e colunas:', tabela.shape)
print('Colunas:', tabela.columns.tolist())
print('Valores ausentes por coluna:')
display(tabela.isna().sum().to_frame('quantidade_ausente'))

## 6. Criar colunas calculadas
Vamos calcular o tamanho da sequência e o percentual de bases G e C.

$$GC\% = \frac{G + C}{\text{tamanho da sequência}} \times 100$$

In [ ]:
tabela['sequencia'] = tabela['sequencia'].str.upper()
tabela['tamanho'] = tabela['sequencia'].str.len()
tabela['quantidade_gc'] = (
    tabela['sequencia'].str.count('G') + tabela['sequencia'].str.count('C')
)
tabela['gc_percentual'] = (
    100 * tabela['quantidade_gc'] / tabela['tamanho']
).round(2)

display(tabela[['id_amostra', 'grupo', 'sequencia', 'tamanho', 'gc_percentual']].head(10))

## 7. Filtrar registros
O filtro seleciona somente as linhas que atendem a uma condição.

In [ ]:
resposta_alta = tabela[tabela['resposta_percentual'] >= 40]
display(resposta_alta[['id_amostra', 'grupo', 'dia_coleta', 'resposta_percentual']])

## 8. Criar uma tabela-resumo
`groupby()` reúne os registros por grupo. `agg()` calcula medidas para cada grupo.

In [ ]:
resumo = (
    tabela.groupby('grupo', as_index=False)
    .agg(
        numero_amostras=('id_amostra', 'count'),
        tamanho_medio=('tamanho', 'mean'),
        gc_medio=('gc_percentual', 'mean'),
        resposta_media=('resposta_percentual', 'mean')
    )
    .round(2)
)
display(resumo)

## 9. Gráfico de barras
Use barras para comparar categorias.

In [ ]:
resumo.plot(
    x='grupo',
    y='resposta_media',
    kind='bar',
    color=['#2563EB', '#10B981', '#F97316'],
    legend=False
)
plt.title('Resposta média por grupo')
plt.xlabel('Grupo')
plt.ylabel('Resposta média (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Gráfico de linhas
Use linhas quando houver uma ordem temporal.

In [ ]:
serie_temporal = tabela.pivot_table(
    index='dia_coleta',
    columns='grupo',
    values='resposta_percentual',
    aggfunc='mean'
)
serie_temporal.plot(marker='o')
plt.title('Resposta ao longo dos dias')
plt.xlabel('Dia de coleta')
plt.ylabel('Resposta média (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Histograma e dispersão
O histograma mostra a distribuição. A dispersão relaciona duas variáveis numéricas.

In [ ]:
tabela['qualidade_media'].plot(kind='hist', bins=6, color='#F97316', edgecolor='white')
plt.title('Distribuição da qualidade média')
plt.xlabel('Qualidade média')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

tabela.plot(x='tamanho', y='gc_percentual', kind='scatter', color='#0F766E')
plt.title('Tamanho da sequência e conteúdo GC')
plt.xlabel('Tamanho da sequência (bases)')
plt.ylabel('Conteúdo GC (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Desafios

1. Mostre somente as amostras do grupo `Tratamento_A`.
2. Descubra qual amostra possui a maior sequência.
3. Calcule a temperatura média por laboratório.
4. Crie um gráfico de barras com o conteúdo GC médio por grupo.
5. Investigue os dois valores ausentes e explique por que não devem ser substituídos sem uma regra.
6. Troque as cores e o título de um gráfico sem alterar os dados.

## Encerramento
A análise seguiu o fluxo: **carregar → conhecer → preparar → resumir → visualizar → interpretar**. A interpretação científica exige dados reais, contexto experimental e métodos adequados.